# 教程 1: 简单平均法入门

## 学习目标

通过本教程，你将学会：
1. 理解为什么简单平均能够降低测量噪声
2. 掌握如何在 Python 中实现简单平均和加权平均
3. 学会使用常见的评估指标（RMSE、相关系数、R²等）
4. 认识简单平均方法的局限性

---

## 1. 什么是数据融合？

想象你想测量今天的气温，但是你有三个温度计：
- 🌡️ 温度计 A：读数 20°C（误差较小）
- 🌡️ 温度计 B：读数 22°C（误差中等）
- 🌡️ 温度计 C：读数 19°C（误差较大）

**问题**：真实温度是多少？如何综合利用这三个测量值得到更准确的结果？

这就是**数据融合（Data Fusion）**要解决的问题。在遥感领域，我们常常有多个卫星传感器测量同一个物理量（如土壤湿度、降雨量等），每个传感器都有自己的误差特性。通过融合多个数据源，我们可以获得比单一数据源更准确的估计。

**简单平均法**是最基础的数据融合方法，它假设所有数据源同等重要，直接将它们的平均值作为最终结果。

---

## 2. 环境设置

首先，我们需要导入必要的 Python 库，并设置好环境。

### 为什么要设置路径？
因为我们的 Notebook 文件在 `examples/` 文件夹中，而我们需要导入的 `collocation` 包在上一级目录。通过 `sys.path.append("..")`，我们告诉 Python 去上一级目录查找模块。

In [ ]:
# 设置路径以导入 collocation 包
import sys
import os
sys.path.append("..")  # 回到上一级目录

# 导入数值计算和绘图库
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 设置中文字体支持（让图表能显示中文）
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 在 Notebook 中直接显示图表
%matplotlib inline

print("✅ 环境设置完成！")

---

## 3. 数据生成原理

### 3.1 真实信号的构造

在真实世界中，我们无法获得"真值"（例如，真实的土壤湿度）。但在教学环境中，我们可以**人为构造一个真实信号**，然后在其基础上添加噪声来模拟观测数据。

我们的真实信号由三部分组成：

$$
\theta(t) = 10 + 5 \sin(t) + 0.5t
$$

其中：
- **常数项 10**：基准值（baseline）
- **正弦项 $5\sin(t)$**：周期性变化（模拟季节性波动）
- **线性项 $0.5t$**：趋势项（模拟长期变化趋势）

### 3.2 观测数据的构造

每个观测产品（例如卫星传感器）看到的数据是：

$$
X_i(t) = \theta(t) + \epsilon_i(t)
$$

其中：
- $X_i(t)$：第 $i$ 个产品的观测值
- $\theta(t)$：真实信号
- $\epsilon_i(t)$：观测噪声，服从正态分布 $\mathcal{N}(0, \sigma_i^2)$

**关键参数**：
- `n_samples`：时间序列的长度（样本数）
- `n_products`：数据产品的数量（例如 3 个卫星传感器）
- `noise_levels`：每个产品的噪声标准差 $\sigma_i$（噪声越大，数据质量越差）

### 💡 直观理解
- 如果 `noise_levels = [1.0, 1.5, 2.0]`，说明产品 1 最准确，产品 3 最不准确
- 增大 `noise_levels`，你会看到数据点更加分散
- 增大 `n_samples`，你会得到更长的时间序列

In [ ]:
def generate_synthetic_data(n_samples=200, n_products=3, noise_levels=None, seed=42):
    """
    生成合成数据用于演示
    
    参数:
        n_samples: 样本数量（时间序列长度）
        n_products: 产品数量（传感器数量）
        noise_levels: 每个产品的噪声水平（标准差）
        seed: 随机种子（保证结果可重复）
        
    返回:
        truth: 真实信号
        products: 各产品观测值列表
        noise_levels: 使用的噪声水平
    """
    np.random.seed(seed)  # 设置随机种子，保证每次运行结果相同
    
    # 生成真实信号: θ(t) = 10 + 5*sin(t) + 0.5*t
    t = np.linspace(0, 4*np.pi, n_samples)  # 时间点
    truth = 10 + 5 * np.sin(t) + 0.5 * t
    
    # 默认噪声水平
    if noise_levels is None:
        noise_levels = [1.0, 1.5, 2.0][:n_products]
    
    # 生成观测产品: X_i(t) = θ(t) + ε_i(t)
    products = []
    for i, noise_std in enumerate(noise_levels):
        noise = np.random.normal(0, noise_std, n_samples)  # 生成噪声
        product = truth + noise  # 真值 + 噪声 = 观测值
        products.append(product)
        print(f"产品 {i+1}: 噪声标准差 σ = {noise_std:.2f}")
    
    return truth, products, noise_levels

In [ ]:
# 生成数据
print("正在生成合成数据...")
truth, products, noise_levels = generate_synthetic_data(
    n_samples=200,       # 200 个时间点
    n_products=3,        # 3 个数据产品
    noise_levels=[1.0, 1.5, 2.0]  # 不同的噪声水平
)

print(f"\n✅ 数据生成完成！")
print(f"   - 样本数量: {len(truth)}")
print(f"   - 产品数量: {len(products)}")
print(f"   - 真实信号范围: [{truth.min():.2f}, {truth.max():.2f}]")

In [ ]:
# 👀 让我们看看前几个数据点
print("\n前 5 个时间点的数据：")
print("-" * 60)
print(f"{'时刻':<8} {'真值':<12} {'产品1':<12} {'产品2':<12} {'产品3':<12}")
print("-" * 60)
for i in range(5):
    print(f"{i:<8} {truth[i]:<12.4f} {products[0][i]:<12.4f} {products[1][i]:<12.4f} {products[2][i]:<12.4f}")
print("-" * 60)

---

## 4. 简单平均方法原理

### 4.1 算术平均

给定 $N$ 个观测产品 $X_1, X_2, \ldots, X_N$，**简单算术平均**的公式是：

$$
\bar{X} = \frac{1}{N} \sum_{i=1}^{N} X_i
$$

### 4.2 为什么平均能降低噪声？

假设每个观测的误差是**独立**且**零均值**的：

$$
\text{Var}(\epsilon_i) = \sigma_i^2
$$

那么平均值的方差是：

$$
\text{Var}(\bar{X}) = \text{Var}\left(\frac{1}{N} \sum_{i=1}^{N} X_i\right) = \frac{1}{N^2} \sum_{i=1}^{N} \text{Var}(X_i) = \frac{1}{N^2} \sum_{i=1}^{N} \sigma_i^2
$$

**如果所有产品噪声相同**（$\sigma_i = \sigma$），则：

$$
\text{Var}(\bar{X}) = \frac{\sigma^2}{N}
$$

**结论**：
- 平均 $N$ 个独立观测，误差方差减小为原来的 $\frac{1}{N}$
- 误差标准差减小为原来的 $\frac{1}{\sqrt{N}}$
- 例如：平均 9 个观测，噪声可以降低到原来的 $\frac{1}{3}$

### 4.3 加权平均

如果我们知道每个产品的噪声水平不同，可以给质量更好的数据**更大的权重**：

$$
\bar{X}_{\text{weighted}} = \sum_{i=1}^{N} w_i X_i, \quad \text{其中} \quad \sum_{i=1}^{N} w_i = 1
$$

**最优权重**（逆方差加权）：

$$
w_i = \frac{1/\sigma_i^2}{\sum_{j=1}^{N} 1/\sigma_j^2}
$$

即：**噪声越小的数据，权重越大**。

In [ ]:
def simple_average(products, weights=None):
    """
    计算多个产品的简单平均或加权平均
    
    参数:
        products: 产品列表，每个产品是一维数组
        weights: 权重列表，如果为None则使用等权重
        
    返回:
        averaged: 平均结果
    """
    products_array = np.array(products)
    
    if weights is None:
        # 简单算术平均: X̄ = (1/N) * Σ X_i
        averaged = np.mean(products_array, axis=0)
        print("使用简单算术平均（等权重）")
    else:
        # 加权平均: X̄ = Σ w_i * X_i
        weights = np.array(weights)
        weights = weights / weights.sum()  # 归一化权重，确保和为1
        averaged = np.sum(products_array * weights[:, np.newaxis], axis=0)
        print(f"使用加权平均，归一化后的权重: {weights}")
    
    return averaged

In [ ]:
# 计算简单平均
print("正在计算简单平均...")
averaged = simple_average(products)
print("\n✅ 简单平均计算完成！")
print(f"   平均值范围: [{averaged.min():.2f}, {averaged.max():.2f}]")

---

## 5. 性能评估指标

为了量化融合效果，我们需要计算一些评估指标。这些指标可以告诉我们：估计值与真值有多接近？

### 常用指标

1. **RMSE (Root Mean Square Error，均方根误差)**
   $$
   \text{RMSE} = \sqrt{\frac{1}{n} \sum_{t=1}^{n} (X_t - \theta_t)^2}
   $$
   - 单位与数据相同
   - 越小越好
   - 对大误差更敏感

2. **相关系数 (Correlation Coefficient)**
   $$
   r = \frac{\text{Cov}(X, \theta)}{\sigma_X \sigma_\theta}
   $$
   - 取值范围：[-1, 1]
   - 越接近 1 越好
   - 衡量线性相关性

3. **R² (决定系数)**
   $$
   R^2 = 1 - \frac{\sum (\theta - X)^2}{\sum (\theta - \bar{\theta})^2}
   $$
   - 取值范围：(-∞, 1]
   - 越接近 1 越好
   - 表示模型解释的方差比例

4. **MAE (Mean Absolute Error，平均绝对误差)**
   $$
   \text{MAE} = \frac{1}{n} \sum_{t=1}^{n} |X_t - \theta_t|
   $$
   - 对异常值不敏感
   - 越小越好

5. **Bias (偏差)**
   $$
   \text{Bias} = \frac{1}{n} \sum_{t=1}^{n} (X_t - \theta_t)
   $$
   - 衡量系统性误差
   - 理想情况下应接近 0

In [ ]:
def calculate_metrics(estimated, truth):
    """
    计算评估指标
    
    参数:
        estimated: 估计值
        truth: 真实值
        
    返回:
        metrics: 指标字典
    """
    # RMSE: 均方根误差
    rmse = np.sqrt(np.mean((estimated - truth)**2))
    
    # 相关系数
    correlation = np.corrcoef(estimated, truth)[0, 1]
    
    # R²: 决定系数
    ss_res = np.sum((truth - estimated)**2)  # 残差平方和
    ss_tot = np.sum((truth - np.mean(truth))**2)  # 总平方和
    r2 = 1 - (ss_res / ss_tot)
    
    # MAE: 平均绝对误差
    mae = np.mean(np.abs(estimated - truth))
    
    # Bias: 偏差
    bias = np.mean(estimated - truth)
    
    metrics = {
        'RMSE': rmse,
        'Correlation': correlation,
        'R2': r2,
        'MAE': mae,
        'Bias': bias
    }
    
    return metrics

In [ ]:
# 计算各产品和平均结果的性能指标
print("正在计算性能指标...\n")
print("=" * 90)
print(f"{'方法':<15} {'RMSE':<12} {'相关系数':<12} {'R²':<12} {'MAE':<12} {'偏差':<12}")
print("=" * 90)

all_metrics = []

# 各产品的指标
for i, product in enumerate(products):
    metrics = calculate_metrics(product, truth)
    all_metrics.append(metrics)
    print(f"产品 {i+1:<10} {metrics['RMSE']:<12.4f} {metrics['Correlation']:<12.4f} "
          f"{metrics['R2']:<12.4f} {metrics['MAE']:<12.4f} {metrics['Bias']:<12.4f}")

# 简单平均的指标
avg_metrics = calculate_metrics(averaged, truth)
all_metrics.append(avg_metrics)
print("-" * 90)
print(f"{'简单平均':<15} {avg_metrics['RMSE']:<12.4f} {avg_metrics['Correlation']:<12.4f} "
      f"{avg_metrics['R2']:<12.4f} {avg_metrics['MAE']:<12.4f} {avg_metrics['Bias']:<12.4f}")
print("=" * 90)

In [ ]:
# 性能改进分析
print("\n📊 性能改进分析")
print("-" * 50)

# 相对于最好的单个产品
best_product_rmse = min(m['RMSE'] for m in all_metrics[:-1])
improvement = (best_product_rmse - avg_metrics['RMSE']) / best_product_rmse * 100

print(f"最优单产品 RMSE: {best_product_rmse:.4f}")
print(f"简单平均 RMSE:   {avg_metrics['RMSE']:.4f}")
print(f"相对改进:        {improvement:.2f}%")
print("-" * 50)

if improvement > 0:
    print("✅ 简单平均优于任何单一产品！")
else:
    print("⚠️  简单平均未能超越最优产品（可能是噪声差异太大）")

---

## 6. 可视化结果

让我们通过图表直观地看到简单平均的效果。

In [ ]:
# 创建可视化图表
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# ============================================================
# 上图：时间序列比较
# ============================================================
ax1 = axes[0]
t = np.arange(len(truth))

# 绘制真实信号（粗黑线）
ax1.plot(t, truth, 'k-', linewidth=2.5, label='真实信号 θ(t)', zorder=10)

# 绘制各产品（细彩色线）
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, (product, noise) in enumerate(zip(products, noise_levels)):
    ax1.plot(t, product, color=colors[i], alpha=0.6, linewidth=1.5, 
            label=f'产品 {i+1} (σ={noise:.1f}, RMSE={all_metrics[i]["RMSE"]:.2f})')

# 绘制简单平均结果（粗红线）
ax1.plot(t, averaged, 'r-', linewidth=2.5, 
        label=f'简单平均 (RMSE={avg_metrics["RMSE"]:.2f})', zorder=9)

ax1.set_xlabel('时间步 (t)', fontsize=13)
ax1.set_ylabel('观测值', fontsize=13)
ax1.set_title('时间序列比较：简单平均 vs 单一产品', fontsize=15, fontweight='bold')
ax1.legend(loc='upper left', fontsize=11, framealpha=0.9)
ax1.grid(True, alpha=0.3, linestyle='--')

# ============================================================
# 下图：误差分布箱线图
# ============================================================
ax2 = axes[1]

errors = []
labels = []

# 各产品误差
for i, product in enumerate(products):
    error = product - truth
    errors.append(error)
    labels.append(f'产品 {i+1}')

# 平均结果误差
avg_error = averaged - truth
errors.append(avg_error)
labels.append('简单平均')

# 绘制箱线图
bp = ax2.boxplot(errors, labels=labels, patch_artist=True, widths=0.6)

# 设置颜色
for i, patch in enumerate(bp['boxes']):
    if i < len(products):
        patch.set_facecolor(colors[i])
        patch.set_alpha(0.7)
    else:
        patch.set_facecolor('red')
        patch.set_alpha(0.8)

# 添加零线（理想情况下误差应该围绕0分布）
ax2.axhline(y=0, color='k', linestyle='--', linewidth=1.5, alpha=0.7, label='零误差线')
ax2.set_ylabel('误差 (观测值 - 真值)', fontsize=13)
ax2.set_title('误差分布对比：简单平均显著降低误差波动', fontsize=15, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y', linestyle='--')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("\n📈 图表说明：")
print("   - 上图：可以看到简单平均（红线）比单一产品更接近真实信号（黑线）")
print("   - 下图：箱线图显示简单平均的误差分布更窄，说明波动更小")

---

## 7. 加权平均：利用先验知识

如果我们**事先知道**每个产品的噪声水平（在实际应用中可以通过历史数据或三重配准法估计），可以使用**逆方差加权**来进一步改进。

### 逆方差加权公式

$$
w_i = \frac{1/\sigma_i^2}{\sum_{j=1}^{N} 1/\sigma_j^2}
$$

**直观理解**：
- 噪声小的数据（$\sigma_i$ 小）→ $1/\sigma_i^2$ 大 → 权重大
- 噪声大的数据（$\sigma_i$ 大）→ $1/\sigma_i^2$ 小 → 权重小

In [ ]:
# 计算逆方差权重
print("正在计算加权平均（逆方差加权）...\n")

# 权重 = 1 / σ²
weights = [1.0 / (n**2) for n in noise_levels]
print(f"原始权重（未归一化）: {[f'{w:.4f}' for w in weights]}")

# 归一化权重（确保和为1）
normalized_weights = np.array(weights) / sum(weights)
print(f"归一化权重:          {[f'{w:.4f}' for w in normalized_weights]}")
print()

# 计算加权平均
weighted_avg = simple_average(products, weights=weights)
weighted_metrics = calculate_metrics(weighted_avg, truth)

print("\n📊 加权平均 vs 简单平均")
print("-" * 60)
print(f"{'方法':<20} {'RMSE':<12} {'相关系数':<12} {'R²':<12}")
print("-" * 60)
print(f"{'简单平均':<20} {avg_metrics['RMSE']:<12.4f} {avg_metrics['Correlation']:<12.4f} {avg_metrics['R2']:<12.4f}")
print(f"{'加权平均':<20} {weighted_metrics['RMSE']:<12.4f} {weighted_metrics['Correlation']:<12.4f} {weighted_metrics['R2']:<12.4f}")
print("-" * 60)

# 改进幅度
improvement_weighted = (avg_metrics['RMSE'] - weighted_metrics['RMSE']) / avg_metrics['RMSE'] * 100
print(f"\n加权平均相对简单平均的改进: {improvement_weighted:.2f}%")

if improvement_weighted > 0:
    print("✅ 加权平均优于简单平均！")
else:
    print("⚠️  加权平均未能超越简单平均（可能是样本量不足或权重不准确）")

---

## 8. 总结与思考

### ✅ 你学到了什么？

1. **简单平均是最基础的数据融合方法**
   - 实现简单，计算快速
   - 不需要复杂的统计假设
   - 适用于快速初步分析

2. **平均能降低随机误差**
   - 理论上，误差方差降低为 $1/N$
   - 前提是误差独立且零均值

3. **加权平均可以利用先验知识**
   - 逆方差加权给质量更好的数据更大权重
   - 可以进一步降低误差

### ⚠️ 简单平均的局限性

1. **无法估计误差结构**
   - 简单平均只给出融合结果，不知道融合后的误差有多大
   - 无法量化不确定性

2. **无法处理系统性偏差**
   - 如果所有产品都存在相同方向的偏差，平均无法消除
   - 例如：3个温度计都偏高2度，平均后仍然偏高2度

3. **假设所有产品同等重要**（简单平均）
   - 实际上不同数据源质量可能差异很大
   - 需要更高级的方法（如三重配准法 TC、信息向量法 IVD）来估计权重

### 🚀 下一步学习

对于更复杂的场景，建议使用：
- **IVD (Information Vector Dual)**：两数据源情况
- **TC (Triple Collocation)**：三数据源情况，可以估计误差方差
- **EIVD / EC**：多数据源情况

---

## 💡 思考题与动手尝试

### 思考题 1：样本量的影响
**问题**：根据理论，增加样本量应该能得到更稳定的估计。请修改 `n_samples` 参数，观察 RMSE 的变化。

**尝试**：
- 将 `n_samples` 从 200 改为 50，观察结果
- 将 `n_samples` 改为 5000，观察结果
- 多次运行（修改 `seed` 参数），观察结果的稳定性

### 思考题 2：噪声水平的影响
**问题**：如果三个产品的噪声水平差异很大会怎样？

**尝试**：
- 修改 `noise_levels = [0.5, 5.0, 10.0]`（差异很大）
- 观察简单平均和加权平均的差距是否变大
- 思考：为什么差距会变大？

### 思考题 3：产品数量的影响
**问题**：增加产品数量能否进一步降低误差？

**尝试**：
- 修改 `n_products = 5` 和 `noise_levels = [1.0, 1.2, 1.5, 1.8, 2.0]`
- 观察 RMSE 是否进一步降低
- 验证理论：误差方差 ∝ 1/N

### 动手实验：添加系统性偏差
**任务**：修改数据生成代码，给每个产品添加一个固定偏差（bias）。

**代码提示**：
```python
biases = [0.5, -0.3, 0.2]  # 系统性偏差
for i, (noise_std, bias) in enumerate(zip(noise_levels, biases)):
    noise = np.random.normal(0, noise_std, n_samples)
    product = truth + noise + bias  # 添加偏差
    products.append(product)
```

**观察**：
- 简单平均能否消除系统性偏差？
- 查看 Bias 指标的变化

---

## 📚 参考资料

- [Triple Collocation 原理介绍](https://journals.ametsoc.org/view/journals/atot/15/6/1520-0426_1998_015_1695_tttnsw_2_0_co_2.xml)
- [数据融合方法综述](https://www.sciencedirect.com/science/article/pii/S0034425716300013)
- 本项目 GitHub: [Collocation-Analysis](https://github.com/yourusername/Collocation-Analysis)

---

**恭喜你完成了第一个教程！🎉**

你现在已经理解了数据融合的基本概念，并掌握了最简单的融合方法。在后续的教程中，我们将学习更高级的方法，它们可以在不知道真值的情况下估计误差结构。